# ⚡ Day 03a: Activations & Loss Functions — The Complete Guide

---

## 🎯 What You'll Master Today
- All major activations: ReLU, LeakyReLU, GELU, sigmoid, tanh, softmax
- Loss functions: MSE, BCE, Cross-Entropy
- When to use each combination

**Goal:** Know exactly which activation + loss for any problem type.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  ACTIVATION + LOSS — Decision Table                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Problem Type       Output Act.    Loss Function                ║
║  ─────────────────────────────────────────────────────           ║
║  Binary classif.    sigmoid        BCELoss / BCEWithLogitsLoss  ║
║  Multi-class         softmax        CrossEntropyLoss            ║
║  Regression          none (linear)  MSELoss / L1Loss            ║
║                                                                   ║
║  Hidden layers:     ReLU (default), GELU (transformers)         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# 1. All Major Activation Functions — Implementation
# ============================================================

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def tanh_act(z):
    return np.tanh(z)

def relu(z):
    return np.maximum(0, z)

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def elu(z, alpha=1.0):
    return np.where(z > 0, z, alpha * (np.exp(z) - 1))

def gelu(z):
    """Gaussian Error Linear Unit — used in transformers."""
    return 0.5 * z * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

z = np.linspace(-3, 3, 7)
print(f"z:          {z.round(2)}")
print(f"sigmoid:    {sigmoid(z).round(3)}")
print(f"tanh:       {tanh_act(z).round(3)}")
print(f"relu:       {relu(z).round(3)}")
print(f"leaky_relu: {leaky_relu(z).round(3)}")
print(f"gelu:       {gelu(z).round(3)}")

In [ ]:
# ============================================================
# 2. Activation Derivatives (for backprop)
# ============================================================

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)            # max=0.25 at z=0

def tanh_deriv(z):
    return 1 - np.tanh(z)**2       # max=1 at z=0

def relu_deriv(z):
    return (z > 0).astype(float)   # 0 or 1

def leaky_relu_deriv(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)  # never fully 0!

z = np.array([-2, -1, 0, 1, 2])
print("Derivatives at z =", z)
print(f"  sigmoid':     {sigmoid_deriv(z).round(4)}    ← vanishes!")
print(f"  tanh':        {tanh_deriv(z).round(4)}")
print(f"  relu':        {relu_deriv(z)}         ← dead at z<0")
print(f"  leaky_relu':  {leaky_relu_deriv(z)}   ← never dead")

print("\n💡 Vanishing gradient problem:")
print(f"  sigmoid chain (10 layers): {0.25**10:.10f} → gradient dies!")
print(f"  relu chain (10 layers): 1.0 → gradient survives!")

In [ ]:
# ============================================================
# 3. Loss Functions — Implementation
# ============================================================

# --- MSE: Mean Squared Error (Regression) ---
def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true)**2)

def mse_gradient(y_pred, y_true):
    return 2 * (y_pred - y_true) / len(y_true)

# --- BCE: Binary Cross-Entropy ---
def bce_loss(y_pred, y_true):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def bce_gradient(y_pred, y_true):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -(y_true/y_pred - (1-y_true)/(1-y_pred)) / len(y_true)

# --- CE: Cross-Entropy (Multi-class) ---
def cross_entropy_loss(y_pred_probs, y_true_onehot):
    eps = 1e-15
    y_pred_probs = np.clip(y_pred_probs, eps, 1)
    return -np.mean(np.sum(y_true_onehot * np.log(y_pred_probs), axis=1))

# Test MSE
y_true = np.array([3.0, 5.0, 2.0])
y_pred = np.array([2.8, 5.2, 2.1])
print(f"MSE: {mse_loss(y_pred, y_true):.6f}")

# Test BCE
y_true_b = np.array([1, 0, 1, 0])
y_pred_b = np.array([0.9, 0.1, 0.8, 0.3])
print(f"BCE: {bce_loss(y_pred_b, y_true_b):.6f}")

# Test CE
y_true_oh = np.array([[1,0,0], [0,1,0], [0,0,1]])  # one-hot
y_pred_p = np.array([[0.7,0.2,0.1], [0.1,0.8,0.1], [0.2,0.1,0.7]])  # softmax output
print(f"CE:  {cross_entropy_loss(y_pred_p, y_true_oh):.6f}")

In [ ]:
# ============================================================
# 4. Why BCE + Sigmoid Simplifies to (A - y)
# ============================================================

# This is the most important simplification in DL!

# Loss = -[y·log(σ(z)) + (1-y)·log(1-σ(z))]
# dL/dz = σ(z) - y   ← CLEAN!

# Proof (step by step):
# dL/dz = dL/dσ · dσ/dz
# dL/dσ = -(y/σ - (1-y)/(1-σ))
# dσ/dz = σ(1-σ)     ← sigmoid derivative
# dL/dz = -(y/σ - (1-y)/(1-σ)) · σ(1-σ)
#        = -(y(1-σ) - (1-y)σ)
#        = -(y - yσ - σ + yσ)
#        = -(y - σ)
#        = σ - y = A - y  ✓

# Verify numerically
z_test = 1.5
y_test = 1.0
eps = 1e-7

# Analytical
analytical = sigmoid(z_test) - y_test

# Numerical
loss_plus = -y_test * np.log(sigmoid(z_test + eps))
loss_minus = -y_test * np.log(sigmoid(z_test - eps))
numerical = (loss_plus - loss_minus) / (2 * eps)

print(f"Analytical: dL/dz = {analytical:.8f}")
print(f"Numerical:  dL/dz = {numerical:.8f}")
print(f"Match: {abs(analytical - numerical) < 1e-5}")

print("\n💡 This is why output gradient is always just (A - y).")
print("   Same simplification works for softmax + CE → (A - y) per class!")

In [ ]:
# ============================================================
# 5. Activation Comparison — When to Use What
# ============================================================

print("""
ACTIVATION CHEAT SHEET
══════════════════════

┌─────────────┬──────────────┬──────────────────┬──────────────────────────┐
│ Activation  │ Range        │ Use Where        │ Watch Out                │
├─────────────┼──────────────┼──────────────────┼──────────────────────────┤
│ ReLU        │ [0, ∞)       │ Hidden (default) │ Dead neurons (z<0)       │
│ LeakyReLU   │ (-∞, ∞)      │ Hidden           │ α hyperparameter         │
│ GELU        │ ≈(-0.17, ∞)  │ Transformers     │ Slower to compute        │
│ Sigmoid     │ (0, 1)       │ Binary output    │ Vanishing gradient       │
│ Tanh        │ (-1, 1)      │ RNNs, LSTM gates │ Vanishing gradient       │
│ Softmax     │ (0, 1) Σ=1   │ Multi-class out  │ Numerical overflow       │
│ None/Linear │ (-∞, ∞)      │ Regression out   │ No non-linearity         │
└─────────────┴──────────────┴──────────────────┴──────────────────────────┘

LOSS CHEAT SHEET
════════════════

┌────────────────────┬──────────────────┬──────────────────────────┐
│ Loss               │ Problem Type     │ Output Layer             │
├────────────────────┼──────────────────┼──────────────────────────┤
│ MSE (L2)           │ Regression       │ Linear (no activation)   │
│ MAE (L1)           │ Regression       │ Linear (robust to outl.) │
│ BCE                │ Binary classif.  │ Sigmoid                  │
│ BCEWithLogitsLoss  │ Binary classif.  │ None (more stable)       │
│ CrossEntropyLoss   │ Multi-class      │ None (includes softmax)  │
│ NLLLoss            │ Multi-class      │ LogSoftmax               │
└────────────────────┴──────────────────┴──────────────────────────┘
""")

---

## 🗺️ Activations & Loss Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  DECISION FLOW                                                   ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Hidden layers → ReLU (default) / GELU (transformers)           ║
║                                                                   ║
║  Output layer:                                                   ║
║    Regression  → no activation + MSE                            ║
║    Binary      → sigmoid + BCE (or logits + BCEWithLogits)      ║
║    Multi-class → no activation + CrossEntropyLoss               ║
║                  (PyTorch CE includes softmax internally)       ║
║                                                                   ║
║  Numerical stability:                                            ║
║    • Clip sigmoid input to [-500, 500]                          ║
║    • Subtract max before softmax exp                            ║
║    • Add eps=1e-15 before log                                   ║
║    • Use LogSoftmax + NLLLoss = CrossEntropyLoss               ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Why ReLU over sigmoid for hidden layers? | No vanishing gradient (derivative is 0 or 1). Faster to compute. Sparse activations |
| 2 | What is the dying ReLU problem? | Neurons where z<0 always output 0, gradient=0, never update. Fix: LeakyReLU, ELU |
| 3 | Why does PyTorch CE include softmax? | Combining log(softmax(z)) is numerically more stable than separate softmax then log |
| 4 | MSE vs BCE for classification? | BCE penalizes confident wrong predictions much more. MSE gradients vanish near 0 and 1 |
| 5 | What is GELU and why in transformers? | Smooth approximation of ReLU. Probabilistic interpretation. Used in BERT, GPT |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • ReLU = default hidden. GELU = transformers           │
## │  • Binary: sigmoid + BCE. Multi: softmax + CE           │
## │  • Regression: linear + MSE                             │
## │  • sigmoid + BCE gradient = A - y (memorize!)           │
## │  • Always think about numerical stability               │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Optimizers & Regularization** — SGD, Adam, dropout, batch norm